# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

In [10]:
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [11]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [12]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [13]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [14]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [15]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [16]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [17]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 111,
 'tn': 2590,
 'fp': 47,
 'fn': 252,
 'misclassification_rate': 0.09966666666666667,
 'false_positive_rate': 0.01782328403488813,
 'false_negative_rate': 0.6942148760330579}

### Check results on the test set (new data not yet seen by the model)

In [18]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 34,
 'tn': 860,
 'fp': 14,
 'fn': 92,
 'misclassification_rate': 0.106,
 'false_positive_rate': 0.016018306636155607,
 'false_negative_rate': 0.7301587301587301}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I believe that the model is very good at predicting the bot because there Misclassification rate is relatively low, being at .106 for the test set, which makes the model incorrectly classify objects 10.6% of the time, but still be correct 89.4% set of the time. Therefore, with test data, the model is going to be able to accuratly predict a bot 89.4% of the time

### What are potential ramifications of false positives from the model?

A couple of ramifications that may come from a false positives is that someone who isn't a bot is saidto be a bot, this could lead to an issue such as locked out, password breech alerts, and many other issues regarding bot detection. more eimportantly, the more the model gives false positives, the less trustworthy it becomes.

### What are potential ramifications of false negatives from the model?

A couple of ramifications from false negatives are that bots become underdetected, meaning a bot can pass as a human, which defeats the point of security. Similar to the question above, false negatives also make the model less reliable the more it incorrectly guesses.